In [1]:
import sys
from pyprojroot import here

# Ritorna il percorso assoluto della root del progetto
PROJECT_ROOT = str(here()) + "/"
sys.path.append(PROJECT_ROOT)

print(f"La root del progetto è: {PROJECT_ROOT}")

La root del progetto è: /home/cvalentino/SissaUnisaDraftCodes/


In [2]:
from paths import INV_PATH, TP1_PATH, MODEL_PATH, ROCK_PATH

ABS_PATH = PROJECT_ROOT + INV_PATH + TP1_PATH
LOAD_MODEL = PROJECT_ROOT + MODEL_PATH + ROCK_PATH

model_name = "rock"
xdmf_file_name = "rock.xdmf"

load_params = ABS_PATH + "files/pred_parameters.csv"

dbdir = ABS_PATH + "database/"

In [3]:
import numpy as np
import fenics as fe
import pandas as pd

from modelaquisition.msh2xdmf import Msh2Xdmf

from fem_problems.invTP1.finite_element import PoissonFEM
from fem_problems.invTP1.rbnics_pod import PODReduction

[michael-Alienware-Aurora-R15:3078877] shmem: mmap: an error occurred while determining whether or not /tmp/ompi.michael-Alienware-Aurora-R15.1001/jf.0/1253113856/shared_mem_cuda_pool.michael-Alienware-Aurora-R15 could be created.
[michael-Alienware-Aurora-R15:3078877] create_and_attach: unable to create shared memory BTL coordinating structure :: size 134217728 
/home/cvalentino/miniconda3/envs/dtsu/lib/python3.11/site-packages/h5py/__init__.py:36: UserWarning: h5py is running against HDF5 1.14.3 when it was built against 1.14.2, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [4]:
df = pd.read_csv(load_params, sep=";", index_col=0)

mu_pred = [df.iloc[i, 0] for i in range(3)]

In [5]:
# Path to the XDMF file
file_path = ABS_PATH + xdmf_file_name

# Carica la mesh da file XDMF
mesh = fe.Mesh()
with fe.XDMFFile(file_path) as infile:
    infile.read(mesh)

In [6]:
mu_range = [
    (0, 1.),
    (0, 1.),
    (0, 1.),
]

In [7]:
fem_p = PoissonFEM(mesh)
pod = PODReduction(mu_range, fem_p)

In [8]:
pod.load_reduction(directory=dbdir, filename=model_name)

Start LOADING REDUCTION......End LOADING REDUCTION


In [9]:
mu_test = [.1, .2, .5]
mat, _ = pod.solve_rom(mu_test, pod.num_basis)
fom_m, _ = pod.fem_p.solve_fem(mu_test)
real_m, _ = pod.fem_p.exact_solution(mu_test)

print("ERRORE FOM-ESATTA")
e, f = pod.fem_p.compute_error(fom_m, real_m)
print("ERRORE ROM-FOM")
a, b = pod.fem_p.compute_error(mat, fom_m)
print("ERRORE ROM-ESATTA")
c, d = pod.fem_p.compute_error(mat, real_m)

ERRORE FOM-ESATTA
Errore Assoluto: 8.1693e-01
Errore Relativo: 7.6340e-02
ERRORE ROM-FOM
Errore Assoluto: 4.7017e-06
Errore Relativo: 4.5594e-07
ERRORE ROM-ESATTA
Errore Assoluto: 8.1693e-01
Errore Relativo: 7.6339e-02


In [10]:
rock = Msh2Xdmf(ABS_PATH + xdmf_file_name, model_name)

rock.reset_files("_sol_rom")
rock.reset_files("_sol_fom")
rock.reset_files("_sol_an")

rock.add_solution(mat, "_sol_rom", "solution_rom")
rock.add_solution(fom_m, "_sol_fom", "solution_fom")
rock.add_solution(real_m, "_sol_an", "solution_an")

File rock_sol_rom.xdmf cancellato.
File rock_sol_rom.h5 cancellato.
File rock_sol_fom.xdmf cancellato.
File rock_sol_fom.h5 cancellato.
File rock_sol_an.xdmf cancellato.
File rock_sol_an.h5 cancellato.
Aggiunta soluzione statica
Aggiunta soluzione statica
Aggiunta soluzione statica


In [11]:
err = np.abs(mat - real_m) / np.linalg.norm(real_m)

rock.reset_files("_err_rom_an")
rock.add_solution(err, "_err_rom_an", "error")

File rock_err_rom_an.xdmf cancellato.
File rock_err_rom_an.h5 cancellato.
Aggiunta soluzione statica


In [12]:
err_ass_fom_rom = np.abs(mat - fom_m)
err_rel_fom_rom = err_ass_fom_rom / np.linalg.norm(fom_m)

rock.reset_files("_err_fom_rom")
rock.reset_files("_err_rel_fom_rom")
rock.add_solution(err_ass_fom_rom, "_err_fom_rom", "error_fom_rom")
rock.add_solution(err_rel_fom_rom, "_err_rel_fom_rom", "error_rel_fom_rom")

File rock_err_fom_rom.xdmf cancellato.
File rock_err_fom_rom.h5 cancellato.
File rock_err_rel_fom_rom.xdmf cancellato.
File rock_err_rel_fom_rom.h5 cancellato.
Aggiunta soluzione statica
Aggiunta soluzione statica


In [13]:
err_fom_an = np.abs(fom_m - real_m)
err_rel_fom_an = err_fom_an / np.linalg.norm(real_m)

rock.reset_files("_err_fom")
rock.add_solution(err_rel_fom_an, "_err_fom", "error_fom")

File rock_err_fom.xdmf cancellato.
File rock_err_fom.h5 cancellato.
Aggiunta soluzione statica
